# 第9章 图像分割

本章节学习目标：

- 理解本章核心算法的**数学原理**
- 掌握算法的**手写实现**方法
- 学会使用 OpenCV 对应函数进行**工程实践**
- 通过编程练习加深对算法的理解

> **📌 学习建议**：先阅读概念说明，再动手编写代码，最后完成练习


In [ ]:
# -*- coding: utf-8 -*-
# 中文路径兼容的图像读写函数
import numpy as np
import cv2
import os

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False


# 代码实现

我们来编写$k$均值算法，并利用RGB值对图像进行分割。先导入一张图像。


In [ ]:
from sklearn.cluster import KMeans
from matplotlib.image import imread
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 输入一张图像，图像来源于参考文献[5]
image = imread('segmentation.jpeg')[:,:,:3]
# 将RGB值统一到0-1内
if np.max(image)>1:
    image = image / 255

X = image.reshape(-1, image.shape[2])


In [ ]:
# 利用k-means算法进行聚类
segmented_imgs = []

# 设定聚类中心个数
n_cluster= 4
kmeans = KMeans(n_clusters=n_cluster, random_state=42).fit(X)
print(np.unique(kmeans.labels_))

In [ ]:
# 为每一个类别赋予一个对应的颜色，用于展示
def decode_segmap(label_mask, plot=False):
    label_colours = np.asarray([[79, 103, 67], [143, 146, 126], 
                                [129, 94, 64], [52, 53, 55],
                                [96, 84, 70], [164, 149, 129]])
    r = label_mask.copy()
    g = label_mask.copy()
    b = label_mask.copy()
    # 为每个类别赋予对应的R、G、B值
    for ll in range(0, 6):
        r[label_mask == ll] = label_colours[ll, 0]
        g[label_mask == ll] = label_colours[ll, 1]
        b[label_mask == ll] = label_colours[ll, 2]

    rgb = np.zeros((label_mask.shape[0], label_mask.shape[1], 3))
    rgb[:, :, 0] = r
    rgb[:, :, 1] = g
    rgb[:, :, 2] = b

    return rgb


# 获得预测的标签
segmented_img = kmeans.cluster_centers_[kmeans.labels_]
segmented_imgs = decode_segmap(kmeans.labels_.reshape(
                        image.shape[0],image.shape[1]))

# 展示结果
plt.imshow(image[:,:,:3])
plt.title('Original image')
plt.axis('off')
plt.show()

plt.imshow(segmented_imgs.astype(np.uint8))
plt.title('{} center'.format(n_cluster))
plt.axis('off')
plt.show()

再次动手尝试图像分割，并比较设置不同个数的聚类中心对图像分割的影响。

In [ ]:
image = imread('segmentation.jpeg')[:,:,:3]
# 将RGB值统一到0-255内
if np.max(image)>1:
    image = image / 255
    

sp = image.shape
# 增加xy坐标的信息
# 设定一个权重，对坐标信息加权
weight = 2
y = weight * np.array([[i for i in range(sp[1])] 
                       for j in range(sp[0])]) / sp[0] / sp[1]
x = weight * np.array([[j for i in range(sp[1])] 
                       for j in range(sp[0])])/ sp[0] / sp[1]
image = np.append(image, x.reshape(sp[0], sp[1], 1), axis=2)
image = np.append(image, y.reshape(sp[0], sp[1], 1), axis=2)

X = image.reshape(-1, image.shape[2])
segmented_imgs = []

# 将 K 分别设置为6、5、4、3、2
n_colors = (6, 5, 4, 3, 2)
for n_cluster in n_colors:
    kmeans = KMeans(n_clusters=n_cluster, random_state=42).fit(X)
    segmented_img = kmeans.cluster_centers_[kmeans.labels_]
    segmented_imgs.append(decode_segmap(
        kmeans.labels_.reshape(image.shape[0],
                               image.shape[1])).astype(np.uint8))

# 展示结果
plt.figure(figsize=(12,8))
plt.subplot(231)
plt.imshow(image[:,:,:3])
plt.title('Original image')
plt.axis('off')

for idx,n_clusters in enumerate(n_colors):
    plt.subplot(232+idx)
    plt.imshow(segmented_imgs[idx])
    plt.title('{} center'.format(n_clusters))
    plt.axis('off')



接下来，我们将动手编程实现归一化图切割算法。

In [ ]:
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

sample_img = cv_imread('segmentation.jpeg', cv2.IMREAD_GRAYSCALE).astype(
                                                np.float32) / 255
plt.imshow(sample_img, cmap='gray')

我们接着编写权重矩阵。

In [ ]:
def cal_dist_weighted_matrix(size, r, sig_X):
    h, w = size
    X_matrix = np.zeros((h*w, h*w))
    for i in range(h*w):
        for j in range(h*w):
            i_row, i_col = i // w, i % w
            j_row, j_col = j // w, j % w
            dist = np.power(i_row - j_row, 2) + \
                np.power(i_col - j_col, 2)
            
            if np.sqrt(dist) < r:    
                X_matrix[i, j] = np.exp(-dist / sig_X)
    return X_matrix
    
    
def set_weighted_matrix(img, sig_I=0.01, sig_X=5, r=10):
    vec_img = img.flatten()
    F_matrix = np.power(vec_img[None, :] - vec_img[:, None], 2)
    F_matrix /= sig_I
    F_matrix = np.exp(-F_matrix)
    
    X_matrix = cal_dist_weighted_matrix(img.shape, r, sig_X)
    return F_matrix * X_matrix


weighted_matrix = set_weighted_matrix(sample_img)

weighted_matrix

接着求解每一个特征值对应的特征向量。

In [ ]:
from scipy import sparse
from scipy.sparse.linalg import eigs, svds
import skimage

def n_cuts(W, image):
    n = W.shape[0]
    s_D = sparse.csr_matrix((n, n))
    d_i = W.sum(axis=0)
    for i in range(n):
        s_D[i, i] = d_i[i]
    s_W = sparse.csr_matrix(W)
    # print(s_W.shape)
    s_D_nhalf = np.sqrt(s_D).power(-1)
    # print(s_D.shape)

    L = s_D_nhalf @ (s_D - s_W) @ s_D_nhalf
    # print(L.shape)

    _, eigenvalues, eigenvectors = svds(L, which='SM')
    print(eigenvectors.shape)

    for i in range(1, 5):
        print(eigenvectors[i].shape)
        Partition = eigenvectors[i] > \
            np.sum(eigenvectors[i])/len(eigenvectors[i])
        print(Partition.shape)
        skimage.io.imshow(Partition.reshape(image.shape))
        plt.title('Ncut')
        plt.show()

展示相应的分割结果。

In [ ]:
n_cuts(weighted_matrix, sample_img)

<center>
    <span style="color:red">[图片占位 - 需本地生成]</span> width=400>
    <br>
    <div style="color:orange; 
    display: inline-block;
    color: #999;
    padding: 2px;"></div>
</center>


---

## 📝 
练习：本章算法手写实现与扩展



**练习目标**：基于本章所学内容，完成以下实践任务。

**要求**：
1. 手写实现本章的核心算法（不直接调用 OpenCV/PyTorch 对应函数）
2. 使用本章学习的方法处理至少 2 张不同的测试图像
3. 对比手写实现与现成库函数的结果差异
4. 分析算法参数对结果的影响
5. 撰写 200 字以上的实验报告


**💡 小提示**：
- 除 `cv_imread` / `cv_imwrite` 外，不直接调用 OpenCV 高层函数
- 使用 NumPy 进行矩阵运算
- 注意边界处理和数值范围
- 对比手写实现与库函数的结果



<details>
<summary><b>🔑 点击查看完整解决方案</b></summary>

---

### 解决方案详解



In [ ]:
```python
# 本章练习代码框架
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False

# ============================================
# TODO: 在此处手写实现本章核心算法
# ============================================

# 示例框架：
# 1. 数据准备
# img = cv_imread('test_image.jpg')
# gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. 手写算法实现
# def algorithm_manual(input_image, **params):
#     # TODO: 实现算法核心逻辑
#     # 要求：除 OpenCV 读写函数外，其余代码手写
#     return output

# 3. 对比验证
# result_manual = algorithm_manual(gray)
# result_library = cv2.XXX(gray)  # 对应库函数
# diff = np.abs(result_manual.astype(float) - result_library.astype(float))
# print(f"最大差异: {diff.max()}")

# 4. 参数敏感性分析
# for param in [param1, param2, param3]:
#     result = algorithm_manual(gray, param=param)
#     # 可视化结果变化

# 5. 实验报告
print("请完成上述练习并撰写实验报告")
```



### 💻 代码要点解释

1. **图像读取与保存**：使用自定义的 `cv_imread` / `cv_imwrite` 函数，解决 Windows 中文路径下 OpenCV 读写图像失败的问题

2. **算法核心**：手写实现的核心在于**不依赖现成库函数**，而是直接操作像素和矩阵运算

3. **对比验证**：通过与 OpenCV 对应函数的结果进行数值对比，验证手写实现的正确性

4. **参数分析**：调整算法参数，观察输出变化，理解每个参数的物理含义

5. **扩展思考**：尝试将算法应用到自己的图像上，或改进算法（如增加加速技巧）

---

</details>

---
